In [28]:
from google.colab import drive
drive.mount("/content/drive")

import json, os, gc, time
import torch, numpy as np
import soundfile as sf
from pathlib import Path
# !pip install whisperx

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
DRIVE_SEGMENTS_DIR = Path("/content/drive/MyDrive/EFE2025/prueba_Valentin")
OUTPUT_CACHE       = Path("/content/drive/MyDrive/EFE2025/prueba_Valentin.json")

# La carpeta: créala si no existe (parents=True crea también EFE2025 si faltara)
DRIVE_SEGMENTS_DIR.mkdir(parents=True, exist_ok=True)

# El JSON: si no existe, inicialízalo vacío
if not OUTPUT_CACHE.exists():
    OUTPUT_CACHE.write_text('{}', encoding='utf-8')

HF_TOKEN      = "hf_"
MODEL_SIZE    = "large-v3-turbo"
LANGUAGE      = "es"
START_PCT     = 0.0
END_PCT       = 100.0
MIN_SPEAKERS  = 2
MAX_SPEAKERS  = 2
BATCH_SIZE    = 8

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'ninguna'}")

wav_files = sorted(DRIVE_SEGMENTS_DIR.glob("*_FULL.wav"))
print(f"Archivos de audio: {len(wav_files)}")

# Cargar caché existente si existe (para reanudar si se interrumpe)
if OUTPUT_CACHE.exists():
    with open(OUTPUT_CACHE, encoding="utf-8") as f:
        whisper_results = json.load(f)
    print(f"Caché existente: {len(whisper_results)} archivos")
else:
    whisper_results = {}
    print("Caché nuevo")

Dispositivo: cuda
GPU: NVIDIA L4
Archivos de audio: 1
Caché existente: 0 archivos


In [30]:
import whisperx

print(f"Cargando WhisperX '{MODEL_SIZE}'...")
model_w = whisperx.load_model(MODEL_SIZE, DEVICE,
                               compute_type="float16" if DEVICE=="cuda" else "int8",
                               language=LANGUAGE)
model_a, metadata = whisperx.load_align_model(language_code=LANGUAGE, device=DEVICE)
print("✅ Modelos cargados")

Cargando WhisperX 'large-v3-turbo'...
2026-06-11 11:09:55 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`
INFO:lightning.pytorch.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`


✅ Modelos cargados


In [31]:
for wav_path in wav_files:
    # El stem del fichero original es todo antes de _FULL
    stem = wav_path.stem.replace("_FULL", "")

    if stem in whisper_results and whisper_results[stem].get("segments"):
        print(f"  {stem}: ya transcrito — saltando")
        continue

    print(f"\n  ── {stem} ──")
    audio = whisperx.load_audio(str(wav_path))

    # Duración real del segmento (lo que grabamos es el 20-40%)
    seg_duration = len(audio) / 16000
    # Duración total del fichero original: seg_duration / 0.20
    full_duration = seg_duration / ((END_PCT - START_PCT) / 100)
    time_offset   = full_duration * START_PCT / 100

    print(f"  Segmento: {seg_duration/60:.1f} min | offset: {time_offset:.1f}s")
    t0 = time.time()

    result = model_w.transcribe(audio, batch_size=BATCH_SIZE)
    result = whisperx.align(result["segments"], model_a, metadata,
                             audio, DEVICE, return_char_alignments=False)

    whisper_results[stem] = {
        "segments":      result["segments"],
        "full_duration": round(full_duration, 4),
        "time_offset":   round(time_offset, 4),
    }
    print(f"  ✅ {len(result['segments'])} segmentos ({time.time()-t0:.1f}s)")

    # Guardar en Drive tras cada fichero (para no perder progreso)
    with open(OUTPUT_CACHE, "w", encoding="utf-8") as f:
        json.dump(whisper_results, f, ensure_ascii=False, indent=1)

del model_w, model_a
gc.collect()
torch.cuda.empty_cache() if DEVICE == "cuda" else None
print(f"\n✅ Transcripción completada: {len(whisper_results)} archivos")


  ── S1_01 ──
  Segmento: 5.4 min | offset: 0.0s


/usr/local/lib/python3.12/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(


  ✅ 45 segmentos (8.8s)

✅ Transcripción completada: 1 archivos


In [32]:
from huggingface_hub import login as hf_login
from pyannote.audio import Pipeline as PyannotePipeline
import whisperx, pandas as pd

hf_login(token=HF_TOKEN, add_to_git_credential=False)

try:
    diar_pipeline = PyannotePipeline.from_pretrained("pyannote/speaker-diarization-3.1")
    print("Modelo: pyannote/speaker-diarization-3.1")
except Exception as e:
    diar_pipeline = PyannotePipeline.from_pretrained("pyannote/speaker-diarization-community-1")
    print(f"Usando community-1 ({e.__class__.__name__})")
diar_pipeline.to(torch.device(DEVICE))
print("✅ Pyannote listo")

Modelo: pyannote/speaker-diarization-3.1
✅ Pyannote listo


In [33]:
def run_diarization(wav_path, n_min, n_max):
    y, sr = sf.read(str(wav_path), dtype="float32")
    if y.ndim == 1:
        y = y[np.newaxis, :]          # (1, time)
    elif y.ndim == 2:
        y = y.T                       # soundfile devuelve (time, channels) → trasponemos a (channels, time)

    # ⚠️  Pasar en CPU — pyannote gestiona el device internamente
    waveform = torch.from_numpy(y.copy()).float()   # .copy() evita el error de array no-contiguo
    audio_in = {"waveform": waveform, "sample_rate": sr}

    kwargs = {}
    if n_min == n_max:
        kwargs["num_speakers"] = n_min
    else:
        if n_min: kwargs["min_speakers"] = n_min
        if n_max: kwargs["max_speakers"] = n_max

    t0 = time.time()
    result = diar_pipeline(audio_in, **kwargs)
    annotation = (result.exclusive_speaker_diarization
                  if hasattr(result, "exclusive_speaker_diarization") else result)
    rows = [{"start": round(t.start, 4), "end": round(t.end, 4), "speaker": spk}
            for t, _, spk in annotation.itertracks(yield_label=True)]
    print(f"    {time.time()-t0:.1f}s → {len(rows)} segmentos")
    return pd.DataFrame(rows)

for wav_path in wav_files:
    stem = wav_path.stem.replace("_FULL", "")
    if stem not in whisper_results: continue

    # Saltar si ya tiene speakers
    if any(s.get("speaker") for s in whisper_results[stem]["segments"]):
        print(f"  {stem}: ya diarizado — saltando")
        continue

    print(f"  Diarizando {stem}...")
    try:
        diarize_df = run_diarization(wav_path, MIN_SPEAKERS, MAX_SPEAKERS)
        whisper_results[stem]["segments"] = whisperx.assign_word_speakers(
            diarize_df, {"segments": whisper_results[stem]["segments"]}
        )["segments"]
        spks = sorted(set(s.get("speaker","") for s in whisper_results[stem]["segments"]
                          if s.get("speaker")))
        print(f"  ✅ {len(spks)} hablante(s): {spks}")
    except Exception as e:
        print(f"  ❌ Error: {e}")

    # Guardar tras cada fichero
    with open(OUTPUT_CACHE, "w", encoding="utf-8") as f:
        json.dump(whisper_results, f, ensure_ascii=False, indent=1)

  Diarizando S1_01...


/usr/local/lib/python3.12/dist-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1839.)
  std = sequences.std(dim=-1, correction=1)


    8.9s → 87 segmentos
  ✅ 2 hablante(s): ['SPEAKER_00', 'SPEAKER_01']


In [34]:
# !pip install praatio -q  # añade esto a CELDA 1 si no está

from praatio import textgrid as tg_module
from praatio.data_classes.interval_tier import IntervalTier

TG_OUTPUT_DIR = Path("/content/drive/MyDrive/EFE2025/textgrids_whisper")
TG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def sanitize(t):
      return t.replace('"', '""').strip()

for stem, data in whisper_results.items():
      out_path = TG_OUTPUT_DIR / f"{stem}.TextGrid"
      if out_path.exists():
          print(f"  {stem}: TextGrid ya existe — saltando")
          continue

      segs      = data["segments"]
      fd        = data["full_duration"]
      to        = data["time_offset"]   # inicio del recorte en el timeline original

      tg = tg_module.Textgrid()

      # ── Tier 1: transcripción Whisper completa ────────────────────────────────
      whisper_entries = []
      for seg in segs:
          s = round(max(0.0, seg["start"]) + to, 4)
          e = round(min(fd,  seg["end"])   + to, 4)
          if e > s:
              whisper_entries.append((s, e, sanitize(seg.get("text", "").strip())))
      whisper_entries.sort(key=lambda x: x[0])
      tg.addTier(IntervalTier("transcripcion_whisper",
                              whisper_entries, minT=0, maxT=fd))

      # ── Tiers por hablante ────────────────────────────────────────────────────
      speakers = sorted(set(
          seg.get("speaker", "UNKNOWN") or "UNKNOWN"
          for seg in segs
      ))
      for spk in speakers:
          spk_entries = []
          for seg in segs:
              if (seg.get("speaker") or "UNKNOWN") != spk:
                  continue
              s = round(max(0.0, seg["start"]) + to, 4)
              e = round(min(fd,  seg["end"])   + to, 4)
              if e > s:
                  spk_entries.append((s, e, sanitize(seg.get("text", "").strip())))
          spk_entries.sort(key=lambda x: x[0])
          if spk_entries:
              tg.addTier(IntervalTier(f"whisper_{spk}",
                                      spk_entries, minT=0, maxT=fd))

      try:
          tg.save(str(out_path), format="long_textgrid", includeBlankSpaces=True)
          print(f"  ✅ {stem}: {len(whisper_entries)} segmentos, {len(speakers)} hablantes")
      except Exception as e:
          print(f"  ⚠️   {stem}: error al guardar ({e})")

print(f"\n✅ TextGrids en: {TG_OUTPUT_DIR}")
print(f"   Total: {len(list(TG_OUTPUT_DIR.glob('*.TextGrid')))} ficheros")

  ✅ S1_01: 45 segmentos, 2 hablantes

✅ TextGrids en: /content/drive/MyDrive/EFE2025/textgrids_whisper
   Total: 212 ficheros


In [35]:
total = len(whisper_results)
con_spk = sum(1 for d in whisper_results.values()
              if any(s.get("speaker") for s in d["segments"]))

print(f"\n{'='*50}")
print(f"✅ COMPLETADO")
print(f"   Archivos transcritos:  {total}")
print(f"   Con diarización:       {con_spk}")
print(f"   Guardado en Drive:     {OUTPUT_CACHE}")
print(f"{'='*50}")
print(f"""
SIGUIENTE PASO (en tu ordenador):
  1. Descarga '{OUTPUT_CACHE.name}' de Drive
  2. Cópialo a:
     resultados_81_180/whisper_cache.json
  3. Crea el fichero de metadatos:
     echo "large-v3-turbo_20.0_40.0" > resultados_81_180/whisper_cache_meta.txt
  4. Lanza el pipeline:
     printf "0\\n" | python run_pipeline.py --input batch_81_180 --output resultados_81_180
""")

# Descarga directa en Colab (alternativa a Drive)
from google.colab import files
files.download(str(OUTPUT_CACHE))


✅ COMPLETADO
   Archivos transcritos:  1
   Con diarización:       1
   Guardado en Drive:     /content/drive/MyDrive/EFE2025/prueba_Valentin.json

SIGUIENTE PASO (en tu ordenador):
  1. Descarga 'prueba_Valentin.json' de Drive
  2. Cópialo a:
     resultados_81_180/whisper_cache.json
  3. Crea el fichero de metadatos:
     echo "large-v3-turbo_20.0_40.0" > resultados_81_180/whisper_cache_meta.txt
  4. Lanza el pipeline:
     printf "0\n" | python run_pipeline.py --input batch_81_180 --output resultados_81_180



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>